In [1]:
!pip install kafka-python-ng

  Using cached kafka_python_ng-2.2.3-py2.py3-none-any.whl (232 kB)


In [2]:
import pymongo
import config
from utils import filter_row
from pymongo import MongoClient
import csv
from datetime import datetime

# mongodb setup
client = MongoClient(config.IP_ADDRESS, 27017)

# create a new database for each run
client.drop_database(config.DB_NAME)
db = client[config.DB_NAME]

# Part 1 : MongoDB Data Model (1.1, 1.2)

## Data Exploration

In [3]:
import pandas as pd

In [4]:
vehicles_df = pd.read_csv("../data/vehicle.csv")

In [5]:
vehicles_df.groupby("owner_name").size().sort_values(ascending = False)
# The most number of vehicles owned by one person is 30

owner_name
Liyana bin Nur        30
Liyana bin Aisyah     29
Haziq binti Liyana    27
Kumar a/l Suresh      27
Muhammad bin Azlan    26
                      ..
Foo Mei Wei            1
Foo Mei Liang          1
Foo Liang Yan          1
Foo Jun Xuan           1
Ng Mei Yan             1
Length: 1263, dtype: int64

In [6]:
vehicles_df.groupby("car_plate").size().sort_values(ascending = False)

car_plate
TF 7       3
HN 2       3
OP 7       3
BS 4       3
WG 5       3
          ..
IX 0       1
IX 6       1
IX 6123    1
IX 688     1
ZZY 1      1
Length: 9844, dtype: int64

In [7]:
event_A_df = pd.read_csv("../data/camera_event_A.csv")
event_B_df = pd.read_csv("../data/camera_event_B.csv")
event_C_df = pd.read_csv("../data/camera_event_C.csv")

In [8]:
event_A_df["registration_date"] = pd.to_datetime(event_A_df["timestamp"])
event_B_df["registration_date"] = pd.to_datetime(event_A_df["timestamp"])
event_C_df["registration_date"] = pd.to_datetime(event_A_df["timestamp"])

print(event_A_df["timestamp"].min())
print(event_B_df["timestamp"].min())
print(event_C_df["timestamp"].min())
print(vehicles_df["registration_date"].max())

2024-01-01T08:00:00
2024-01-01T08:00:23.896476
2024-01-01T08:00:47.168827
2010-12-30T15:49:00


##### Note:
- There should be no duplicate car plates
- camera events do not specify vehicle type, just car plate
- Plan : Choose the vehicle with the latest registration date.
- Justification : Earliest camera event is in 2024 and latest vehicle registration date is in 2010,
                  so there will not be any clashes or mismatches


## 1.1 Collection Design

### 1. Vehicle Collection
- Stores registered vehicle information and owner details. Each document represents one vehicle.

#### Schema

```json
{
  "car_plate": "string",
  "owner_name": "string",
  "owner_addr": "string",
  "vehicle_type": "string",
  "registration_date": "ISODate"
}
```


#### Sample Document

```json
{
  "car_plate": "ABC123",
  "owner_name": "Example Name",
  "owner_addr": "Example Address",
  "vehicle_type": "Sedan",
  "registration_date": ISODate("2024-01-01T00:00:00Z")
}
```

#### Indexes
- { "car_plate": 1 } — supports fast lookup by registration plate, assuming each car plate is unique.

### 2. Camera Collection
- Stores camera details such as location, highway position, and enforced speed limit. Each document represents one camera.

#### Schema

```json
{
  "camera_id": "integer",
  "location": {
    "type": "string",
    "coordinates": [
      "number",
      "number"
    ]
  },
  "position": "number",
  "speed_limit": "integer"
}
```

#### Sample Document

```json
{
  "camera_id": 1,
  "location": {
    "type": "Point",
    "coordinates": [102.6601002, 2.157730731]
  },
  "position": 152.5,
  "speed_limit": 110
}
```

#### Indexes
- { "camera_id": 1 } — supports quick camera lookup.
- { "location": "2dsphere" } — supports geospatial queries such as finding nearby cameras.





### 3. Violation Collection
- Stores speeding violations detected for each vehicle. Each document represents all violations for one vehicle on one day.

#### Schema

```json
{
  "car_plate": "string",
  "date": "ISODate",
  "violations": [
    {
      "violation_type": "string",
      "camera_id_start": "integer",
      "camera_id_end": "integer",
      "timestamp_start": "ISODate",
      "timestamp_end": "ISODate",
      "speed_reading": "number"
    }
  ]
}
```

#### Sample Document

```json
{
  "car_plate": "ABC123",
  "date": ISODate("2024-01-01T00:00:00Z"),
  "violations": [
    {
      "violation_type": "INSTANTANEOUS",
      "camera_id_start": 1,
      "camera_id_end": 1,
      "timestamp_start": ISODate("2024-01-01T08:00:04Z"),
      "timestamp_end": ISODate("2024-01-01T08:00:04Z"),
      "speed_reading": 125.0
    },
	{ 
      "violation_type": "AVERAGE",
      "camera_id_start": 1,
      "camera_id_end": 2,
      "timestamp_start": ISODate("2024-05-01T14:30:00Z"),
      "timestamp_end": ISODate("2024-05-01T14:35:00Z"),
      "speed_reading": 115.0
    }
  ]
}
```

#### Indexes
- { "car_plate": 1, "date": 1 } — supports quick searches such as locating all violations for a vehicle on a specific day. This pair should be unique.
- { "date": 1 } — supports quick filtering by date


### 4. Invalid Events Collection
- Stores invalid events filtered out from the main stream, such as missing car_plate, invalid timestamp, unknown camera_id, or negative speed. Each document represents a full row that has invalid attributes.

```json
{
  "batch_id": "integer",
  "camera_id": "integer",
  "producer_id": "string",
  "event_id": "string",
  "car_plate": "string",
  "event_time": "ISODate",
  "speed_reading": "number"
}
```

#### Sample Document

```json
{
  "batch_id": 101,
  "camera_id": 1,
  "producer_id": "A",
  "event_id": "47af5792-7e1b-45fb-a55d-e988c06f3c06",
  "car_plate": "ABC123",
  "event_time": ISODate("2024-01-01T08:00:04Z"),
  "speed_reading": 125.0
}
```

#### Indexes
- { "event_id": 1 } — supports fast duplicate detection and event lookup. This field should be unique.



In [9]:
# Create the collections

vehicles = db.vehicles
cameras = db.cameras
violations = db.violations
invalid_events = db.invalid_events
stream_metrics = db.stream_metrics
stream_latency = db.stream_latency


# Create indices

# vehicles
db.vehicles.create_index([("car_plate", 1)], unique = True)

# cameras
db.cameras.create_index([("camera_id", 1)], unique = True)
db.cameras.create_index([("location", "2dsphere")], unique = True)

# violations
db.violations.create_index([("car_plate", 1), ("date", 1)], unique = True)
db.violations.create_index([("date", 1)])

# invalid events
db.invalid_events.create_index([("event_id", 1)], unique = True)

'event_id_1'

## 1.2 Collection Relationship

The collections are related as follows:
- One Vehicle can have many Violations over time.
- One Camera can detect many Violations.
- A violation belongs to one vehicle and involves one camera (instantaneous) or two cameras (average speed).


Vehicle and Camera data are stored in separate collections because they are relatively stable master data and may be 
updated independently. This avoids unnecessary duplication of owner details or camera metadata in every violation record. Not to mention, the documents in the violations collection are built with the necessary camera information (speed limit) available already (during joining), so no further joining is needed. Even if joining is needed, the cost is negligible due to the small size of the cameras dataset. 

For the Violation collection, embedding is chosen to store violations within a daily vehicle document. Each document contains all violations for one vehicle on one day. Since violations for the same vehicle and date are closely related and are likely to be queried together, embedding improves read efficiency and avoids additional join/lookups. According to a publication, "6 Rules of Thumb for MongoDB Schema Design" (MongoDB, n.d.), referencing should be used for a one-to-many case (approximately hundreds of rows, but not thousands), which is a reasonable assumption for this collection since every document records the violations of a specific vehicle on a particular day (Assume that the number of violations is <= 100 for any specific day). However, for this collecition in particular, the number of reads is significantly more than the number of updates, that is, once a violation is recorded, it will not be changed anymore. So denormalisation of many to one is accepted in this case.

Segmenting the records by day also prevents each document from growing excessively large over time, keeping updates and storage manageable. While this design introduces slight duplication, such as repeated car_plate values across multiple dates, the duplication is minimal and acceptable compared to the performance benefit of faster reads. This structure also works well with the index on { car_plate: 1, date: 1 }, allowing efficient retrieval of violations for a specific vehicle on a particular day.









### References
- MongoDB. (n.d.). 6 rules of thumb for MongoDB schema design. MongoDB Blog. https://www.mongodb.com/company/blog/mongodb/6-rules-of-thumb-for-mongodb-schema-design
- MongoDB. (n.d.). Schema validation. MongoDB Manual. https://www.mongodb.com/docs/manual/core/schema-validation/
- Badshah, A. (n.d.). Data modeling and schema design in MongoDB. Medium. https://afzalbadshah.medium.com/data-modeling-and-schema-design-in-mongodb-24a786b1375d


## Storing Cameras and Vehicles

* Camera and vehicle datasets are loaded into MongoDB collections from CSV files.
* For vehicle records, duplicate car plate entries are resolved by retaining the record with the latest registration date.

In [10]:
# Store cameras
with open("../data/camera.csv", mode="r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        cameras.insert_one(filter_row(row))


# Store vehicles
with open("../data/vehicle.csv", mode="r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        cleaned = filter_row(row)

        existing = vehicles.find_one({
            "car_plate": cleaned["car_plate"]
        })

        # No existing plate -> insert
        if existing is None:
            vehicles.insert_one(cleaned)

        else:
            incoming_date = cleaned["registration_date"]
            existing_date = existing["registration_date"]

            # If there are duplicte rows with the same car plate, choose the one
            # with the latest registration date
            if incoming_date > existing_date:
                vehicles.update_one(
                    {"_id": existing["_id"]},
                    {"$set": cleaned}
                )